# Tutorial 14: EmbeddingStore and `adata.embpy`

This tutorial is the preferred embpy workflow for working with generated biological embeddings after model inference.

It uses tiny synthetic data so you can run everything locally without downloading models or calling resolvers, but the same structure scales to genes, proteins, molecules, cytokines, pathways, morphology, text, and perturbation/action embeddings.

The core path is:

```text
BioEmbedder.embed(...) / EmbeddingResult
        -> EmbeddingStore / .emstore
        -> adata.embpy
        -> analysis, plotting, prompting, and ML-ready action tensors
```

The roles stay deliberately separate:

- **AnnData** remains the experiment container: expression, observations, variables, `.obsm`, `.varm`.
- **`EmbeddingResult`** is the canonical in-memory result from embedding generation: matrix, canonical IDs, aliases, provenance.
- **`EmbeddingStore`** is the reusable embedding universe: genes, proteins, molecules, cytokines, text, and typed relations.
- **`adata.embpy`** connects one experiment to those reusable embeddings, runs quick analyses, reuses `embpy.tl`/`embpy.pl`, emits prompt-ready context, and compiles perturbation/action embeddings for ML.

Generated embeddings still never go into `.X`. `.X` remains expression/count-like data; embeddings live in `.obsm`, `.varm`, linked `.emstore` stores, or structured `.uns` metadata.


## 1. Setup

We import only lightweight dependencies and force a non-interactive matplotlib backend so the notebook is safe to run in CI or from a terminal.

In [ ]:
import tempfile
from pathlib import Path

import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Markdown, display
from anndata import AnnData

from embpy.io.result import EmbeddingProvenance, EmbeddingResult
from embpy.store import EmbeddingStore

np.set_printoptions(precision=3, suppress=True)

## 2. Create a tiny AnnData experiment

Think of this as a miniature perturbation experiment:

- six cells/observations
- three genes/features
- three perturbation conditions: `g1`, `g2`, and the combo `g1+g2`
- expression/count-like data remains in `.X`

In [ ]:
obs = pd.DataFrame(
    {
        "perturbation": ["g1", "g1", "g2", "g2", "g1+g2", "g1+g2"],
        "target_group": ["g1", "g1", "g2", "g2", "combo", "combo"],
        "phenotype_score": [1.0, 1.2, 3.0, 2.8, 2.0, 2.2],
        "cell_type": ["T", "T", "B", "B", "T", "B"],
    },
    index=[f"cell{i}" for i in range(6)],
)
var = pd.DataFrame(index=["g1", "g2", "g3"])
X = np.array(
    [
        [10, 2, 0],
        [11, 1, 0],
        [2, 10, 1],
        [3, 9, 1],
        [6, 6, 2],
        [5, 7, 2],
    ],
    dtype=np.float32,
)
adata = AnnData(X=X, obs=obs, var=var)
adata

## 3. Create canonical embedding results

In normal workflows, `BioEmbedder.embed(...)` handles input normalization, canonicalization, provenance, harmonization, and output routing to AnnData or tables. Internally, the clean in-memory unit is `EmbeddingResult`.

Here we construct small `EmbeddingResult` objects directly so the tutorial stays fast and deterministic. The gene result uses canonical gene IDs from `adata.var_names`, and the cell embedding matrix is aligned to `adata.obs_names`. This alignment is what later lets `adata.embpy` decide whether an embedding belongs in `.obsm`, `.varm`, or a linked store.


In [ ]:
gene_result = EmbeddingResult(
    matrix=np.array(
        [
            [1.0, 0.0],
            [0.0, 1.0],
            [0.25, 0.25],
        ],
        dtype=np.float32,
    ),
    entity_ids=("g1", "g2", "g3"),
    entity_type="gene",
    id_scheme="symbol",
    provenance=EmbeddingProvenance(model="toy_gene_model", pooling="mean"),
    aliases={"g1": {"display_name": "Gene 1"}, "g2": {"display_name": "Gene 2"}},
)

cell_embedding = np.array(
    [
        [1.0, 0.0],
        [0.9, 0.1],
        [0.0, 1.0],
        [0.1, 0.9],
        [0.5, 0.5],
        [0.45, 0.55],
    ],
    dtype=np.float32,
)

gene_result

## 4. Build an `EmbeddingStore`

`EmbeddingStore` is the external reusable embedding universe. It is useful for things that are too large, too shared, or too stable to duplicate into every AnnData object: all genes, all proteins, huge molecule libraries, cytokines, pathways, text descriptions, and relation tables.

The store does not run model inference or canonicalization. It stores already-canonical embeddings and relations, then lets many AnnData experiments reuse the same biological embedding universe.


In [ ]:
store = EmbeddingStore()
store.add_result(gene_result, key="gene:toy_gene_model")
store.add_relation(
    "perturbation_targets_gene",
    pd.DataFrame(
        {
            "source_id": ["g1", "g2", "g1+g2", "g1+g2"],
            "target_id": ["g1", "g2", "g1", "g2"],
        }
    ),
    source_type="perturbation",
    target_type="gene",
)

store.describe()

In [ ]:
store.audit()

## 5. Write/read `.emstore`

The MVP `.emstore` format is a directory:

```text
example.emstore/
  manifest.json
  entities/*.parquet
  embeddings/*/matrix.npy
  embeddings/*/index.parquet
  relations/*.parquet
```

Matrices can be memory-mapped on read with `backed=True`, which matters for large molecule/cytokine/protein universes.

In [ ]:
with tempfile.TemporaryDirectory() as tmp:
    out = store.write(Path(tmp) / "toy.emstore")
    loaded = EmbeddingStore.read(out, backed=True)
    print(out)
    print(type(loaded.embedding("gene:toy_gene_model").matrix))
    display(loaded.describe())

## 6. Connect the store to AnnData with `adata.embpy`

`adata.embpy` keeps a semantic registry in `adata.uns["embpy"]`. The registry is the bridge between experiment data and reusable embedding universes.

Aligned matrices go to AnnData-native slots:

- observation/cell embeddings -> `.obsm`
- gene/feature embeddings -> `.varm`
- large non-aligned reusable embeddings -> linked `EmbeddingStore`

This is the harmonization step: downstream code no longer has to guess which IDs, axes, models, or relations an embedding represents. `.X` is left untouched.


In [ ]:
X_before = adata.X.copy()

adata.embpy.register_store(store)
adata.embpy.register_embedding(
    "X_cells_toy",
    cell_embedding,
    entity_ids=adata.obs_names,
    entity_type="cell",
    id_scheme="obs_name",
)
adata.embpy.register_embedding("X_gene_toy", result=gene_result)

print("obsm:", list(adata.obsm.keys()))
print("varm:", list(adata.varm.keys()))
print("X unchanged:", np.array_equal(adata.X, X_before))
adata.embpy.list_embeddings()

## 7. Audit the registry

`audit()` checks the semantic registry, linked stores, and relation coverage. A clean table means no current issues were found.

In [ ]:
adata.embpy.describe()

In [ ]:
adata.embpy.audit()

## 8. Generate prompt-ready context

`adata.embpy.prompt_context(...)` summarizes the registered embedding universe without dumping matrices or dense action vectors. It includes AnnData shape, `.obsm`/`.varm` keys, registered embeddings, linked stores, relations, conditions, actions, splits, and analyses.

This is one concrete advantage of the registry: an analysis assistant, report, or downstream model can see what analyses are valid before you ask for them, without putting huge embedding tables into the prompt.


In [ ]:
context_dict = adata.embpy.prompt_context(output="dict")
print(context_dict["anndata"])
print(context_dict["embpy"])

prompt_context = adata.embpy.prompt_context(output="markdown")
display(Markdown(prompt_context))


## 9. Aggregate embeddings by biological groups

This wraps reusable `embpy.tl` logic. The accessor only resolves the registered embedding and the AnnData metadata column.

Here we compute perturbation-level centroids from cell-level embeddings.

In [ ]:
centroids = adata.embpy.aggregate("X_cells_toy", by="perturbation")
centroids

## 10. Nearest neighbors and similarity/correlation analyses

`adata.embpy.neighbors(...)` returns a tidy table. Under the hood it delegates to `embpy.tl.nearest_neighbors_table(...)`.

`correlate(...)` compares embedding geometry to another embedding space or to a phenotype column.

In [ ]:
adata.embpy.neighbors("X_cells_toy", query="cell0", k=3)

In [ ]:
adata.embpy.correlate("X_cells_toy", phenotype="phenotype_score")

In [ ]:
# Register a second toy embedding so we can compare spaces.
adata.embpy.register_embedding(
    "X_cells_toy_flipped",
    cell_embedding[:, ::-1],
    entity_ids=adata.obs_names,
    entity_type="cell",
    id_scheme="obs_name",
)
adata.embpy.compare_embeddings(["X_cells_toy", "X_cells_toy_flipped"])

## 11. Reuse existing plotting functions

These wrappers call existing `embpy.pl` functions. The goal is not to create a new plotting ecosystem; it is to make the correct plot easy to call from the semantic registry.

In notebooks, use `display(fig)` before `plt.close(fig)`. Closing first can make VS Code/Jupyter hide the figure because the final expression is already closed.


In [ ]:
fig = adata.embpy.plot_embedding("X_cells_toy", method="pca", color="perturbation")
display(fig)
plt.close(fig)


In [ ]:
fig = adata.embpy.plot_similarity("X_cells_toy", label_col="perturbation", title="Toy cell embedding similarity")
display(fig)
plt.close(fig)


In [ ]:
fig = adata.embpy.plot_model_comparison(["X_cells_toy", "X_cells_toy_flipped"], method="cosine_correlation")
display(fig)
plt.close(fig)


In [ ]:
fig = adata.embpy.plot_diagnostics(["X_cells_toy", "X_cells_toy_flipped"], kind="norms")
display(fig)
plt.close(fig)


## 12. Phenotypic activity

This wraps `embpy.tl.phenotypic_activity`, which computes replicate clustering/activity using chunked cosine similarity.

In a real screen this can be used to ask which perturbations produce consistent phenotypic embeddings.

In [ ]:
activity = adata.embpy.score_activity("X_cells_toy", perturbation_col="perturbation", chunk_size=3)
activity

## 13. Set up conditions and compile action embeddings

This is the perturbation-ML bridge. Instead of asking every model pipeline to re-parse perturbation labels, the registry turns experiment conditions into reusable action vectors.

`setup_conditions(...)` defines stable condition IDs. `compile_actions(...)` uses the linked store and relation table to turn perturbation labels into action vectors. The per-observation matrix is written to `.obsm[output_key]`, while a deduplicated condition-level summary is kept in `adata.uns["embpy"]["actions"]`.

For example, `g1+g2` becomes the mean of the `g1` and `g2` gene embeddings. In a real workflow the target embedding could be ESM protein embeddings, molecule embeddings, cytokine embeddings, pathway embeddings, or text-derived mechanism embeddings.


In [ ]:
conditions = adata.embpy.setup_conditions(condition_key="perturbation", control_values=[])
conditions.head()

In [ ]:
action_table = adata.embpy.compile_actions(
    output_key="X_embpy_action",
    relation="perturbation_targets_gene",
    target_embedding="gene:toy_gene_model",
    perturbation_key="perturbation",
    aggregation="mean",
)

action_table

In [ ]:
print("compiled action embedding shape:", adata.obsm["X_embpy_action"].shape)
print("g1+g2 action vector:", action_table.loc["g1+g2"].to_numpy())

## 14. Leakage-aware splits

This MVP supports deterministic random splits by a biological grouping column. Splitting by `target_group`, for example, avoids mixing the same target group between train and test.

In [ ]:
splits = adata.embpy.make_splits(by="target_group", random_state=1)
for name, idx in splits.items():
    print(name, idx.tolist(), adata.obs.iloc[idx]["target_group"].unique().tolist())

## 15. Make a Torch dataset

The accessor can compile aligned state/action/target tensors for simple ML workflows. This is where the harmonized format becomes useful beyond plotting: the same AnnData object now carries expression state, registered action vectors, split definitions, and provenance.

For this MVP, the dataset is intentionally minimal. It is a bridge for quick prototypes and small models, not a replacement for the richer `world_model` dataloaders. The important contract is alignment: every observation has a state, an action vector, a target, and a stable split membership.

For full world-model training, treat this notebook as the semantic data-prep layer: stores define action universes, `adata.embpy` compiles actions and splits, and the training scripts consume the aligned tensors.


In [ ]:
try:
    dataset = adata.embpy.make_torch_dataset(split=splits["train"], action_key="X_embpy_action")
    sample = dataset[0]
    print(sample.keys())
    print("state", sample["state"].shape)
    print("action", sample["action"].shape)
    print("target", sample["target"].shape)
except ImportError as exc:
    print(exc)

## 16. Mental model

Use this split when designing new workflows:

```text
AnnData
  one experiment: cells, expression, obs/var metadata, aligned embeddings

EmbeddingResult
  one canonical embedding result: matrix, canonical IDs, aliases, provenance

EmbeddingStore
  reusable universe: genes, molecules, proteins, cytokines, pathways, text, relations

adata.embpy
  bridge: registry, audits, prompt context, group summaries, plots, action compilation, ML datasets
```

If a function is general numerical analysis, it should live in `embpy.tl`.
If it is a visualization, it should live in `embpy.pl`.
If it is semantic orchestration across AnnData and stores, it belongs in `adata.embpy`.

That is the value added over plain `.obsm`: not just storing an array, but knowing what it represents, which IDs it uses, which relations connect it to perturbations, which analyses are valid, and how to summarize the whole setup for plotting, prompting, and ML.
